# Extend the simulation: objects, planning, Cosmos, and in-context learning

**Time:** 3–5 hours (live GPU + Isaac + Cosmos)  
**Prerequisites:** Notebooks 00–02, `fanuc_er4ia_sim` bootstrapped, Brev instance running.

## Learning objectives

- Trace how cubes enter the USD scene (`scene_config.py` → `cell.add_sort_objects`)
- Add a **small, gated** planning feature without breaking vision-only semantics
- Run Cosmos (`perceive.py`), inspect prompts and artifacts
- Use **in-context learning** for a novel classification task (not arbitrary 6-DoF pose)
- Execute the full pipeline and validate `execution_report.json`

## Important constraints (from AGENTS.md)

- Six cubes: 2× yellow, 2× blue, 2× red → stack **yellow, yellow, blue, blue, red, red**
- Grasp commands come from **vision**, not simulator ground truth
- Hybrid pose: image geometry measures yaw; Cosmos confirms — do not weaken gates to force success

## Checkpoint

A fresh run directory completes with `task_complete: true`, six verified pickups and placements.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

def repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "fanuc_er4ia_sim" / "run_sort_pipeline.sh").is_file():
            return parent
    raise FileNotFoundError("Could not locate fanuc_er4ia_sim")

ROOT = repo_root()
SIM = ROOT / "fanuc_er4ia_sim"
ICL_DIR = SIM / "assets" / "perception_examples"
RUN_ID = os.environ.get("TUTORIAL_RUN_ID", "tutorial_stack")
RUN = SIM / "runs" / RUN_ID
print("Simulation root:", SIM)
print("Planned run directory:", RUN)

## 1. Preflight (Brev / Linux)

In [ ]:
def sh(cmd: str, *, check: bool = True) -> subprocess.CompletedProcess:
    print("$", cmd)
    return subprocess.run(cmd, shell=True, cwd=SIM, text=True, check=check)

required = ["ISAAC", "COSMOS"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise EnvironmentError(
        "Set environment variables: "
        + ", ".join(missing)
        + ". See tutorials/notebooks/README.md"
    )
for var in ("COSMOS3_NANO_PATH",):
    if not os.environ.get(var):
        print(f"Warning: {var} not set — model discovery may use HF cache only")

# GPU
subprocess.run("nvidia-smi -L", shell=True, check=False)

isaac_py = os.environ["ISAAC"] + "/python.sh"
cosmos = os.environ["COSMOS"]
host = os.environ.get("HOST_PYTHON", str(SIM / ".venv/bin/python"))
for label, path in [("ISAAC", isaac_py), ("COSMOS", cosmos), ("HOST", host)]:
    ok = Path(path).exists() if label != "ISAAC" else Path(isaac_py).exists()
    print(f"{label} launcher exists: {ok}")

In [ ]:
# Cosmos smoke (loads weights — may take minutes)
subprocess.run(
    [
        os.environ["COSMOS"],
        "-c",
        "import torch; print('CUDA:', torch.cuda.is_available())",
    ],
    cwd=SIM,
    check=True,
)

## 2. How objects enter the simulation

1. **`scene_config.py`** — logical layout: `SORT_CUBES`, yaws `SORT_CUBE_YAWS_DEG`, distractors `SORT_VIALS` / `SORT_CAPS`, stack site `CUBE_STACK_XY`.
2. **`cell.add_sort_objects(stage)`** — USD prims `/World/SortCube{i}`, visual from `assets/usd_assets/cube.usda`, invisible box collider, dynamics.

Changing layout, camera, or cube size requires a **new capture** in an empty run directory.

**Exercise (read-only):** Inspect cube count and stack order.

In [ ]:
sys.path.insert(0, str(SIM))
import scene_config as C

print("Cubes on table:", len(C.SORT_CUBES))
for i, (x, y, colour) in enumerate(C.SORT_CUBES):
    yaw = C.SORT_CUBE_YAWS_DEG[i]
    print(f"  SortCube{i}: {colour} at ({x:.3f}, {y:.3f}) yaw={yaw}°")
print("Stack target XY:", C.CUBE_STACK_XY)
print("Stack order:", C.CUBE_STACK_COLOURS)

### Optional layout experiment (requires code edit + full re-run)

To practice adding/changing objects safely:

1. Copy `CUBE_LAYOUT_SEED` to a new value **or** nudge one `(x, y)` in `hackathon_cubes` output (keep six cubes, two per colour).
2. Re-run host tests: `.venv/bin/python -m unittest test_sort_planning test_cube_pose`
3. Build scene: `"$ISAAC/python.sh" build_scene.py --headless --save runs/scene.usd`
4. Run the full pipeline in a **new empty** `runs/<id>/`.

Revert layout changes unless the team wants them permanently.

## 3. Planning: `plan_sort.build_plan`

In [ ]:
from pathlib import Path
import inspect
import plan_sort

print("build_plan signature:")
print(inspect.signature(plan_sort.build_plan))
print()
print(plan_sort.build_plan.__doc__.strip())

### Suggested planning feature (team exercise)

Add a **debug summary** field to `sort_plan.json` without changing pick logic, for example:

- Count of observations by `skip_reason`
- Timestamp and task name

Implementation sketch:

1. Edit `plan_sort.py` `main()` after `build_plan()` returns — add `"tutorial_summary": {...}` to the plan dict.
2. Run host tests (must stay green).
3. Re-run plan stage on an existing run: `.venv/bin/python plan_sort.py --run runs/<id>`

Do **not** pass `ground_truth.json` into `build_plan` or relax confidence/pose gates.

## 4. Working with Cosmos

In [ ]:
perceive_src = (SIM / "perceive.py").read_text(encoding="utf-8")
for name in ("STACK_PROMPT", "CUBE_POSE_PROMPT", "def ask(", "def load_model("):
    idx = perceive_src.find(name if not name.startswith("def") else name.split()[1])
    print(name, "found at char", idx)

### Run perception on a captured run

After capture exists (`world_camera.png`, `camera.json`):

```bash
cd fanuc_er4ia_sim
"$COSMOS" perceive.py --run runs/<id> --stack-cubes
```

Inspect outputs:

| File | Meaning |
|------|---------|
| `detections.json` | Parsed cube list |
| `detections_raw.txt` | Raw model reply |
| `cube_pose_crops/` | Hybrid pose crops + measurements |
| `perception_evaluation.json` | Error vs ground truth (evaluation only) |

In [ ]:
def summarize_run(run_dir: Path):
    if not run_dir.is_dir():
        print("Run directory not found — run pipeline first")
        return
    for name in (
        "detections.json",
        "sort_plan.json",
        "execution_report.json",
        "perception_evaluation.json",
    ):
        p = run_dir / name
        if p.is_file():
            data = json.loads(p.read_text())
            print(f"=== {name} ===")
            if name == "execution_report.json":
                print(
                    "task_complete:",
                    data.get("task_complete"),
                    "pickups:",
                    data.get("verified_pickups"),
                    "placements:",
                    data.get("verified_placements"),
                )
            elif name == "detections.json":
                objs = data.get("objects", [])
                print("objects:", len(objs))
            else:
                print("keys:", list(data.keys())[:8])
        else:
            print(f"(missing) {name}")

summarize_run(RUN)

## 5. In-context learning for novel objects

Cosmos supports multi-turn **examples** in `perceive.ask(model, processor, image, prompt, examples=[...])`.

The stock **cap orientation** path uses `cap_examples()` with PNGs under `assets/perception_examples/` (wide vs narrow end up). Those files are **not** bundled — you create them from your own captures.

**Scope:** ICL here adapts **semantics/classification** (e.g. cap orientation, distractor vs cube). **Quantitative cube yaw** still comes from the image fit + hybrid gate in `inspect_cube_poses`, not from few-shot angle regression (`docs/MODEL_NOTES.md`).

### Exercise: build two-shot cap orientation examples

1. From a table capture, crop two caps with clear orientation differences.
2. Save as `assets/perception_examples/wide_end_up.png` and `narrow_end_up.png`.
3. Run `"$COSMOS" perceive.py --run runs/<id> --sort` and compare orientations in `detections.json`.
4. Compare **zero-shot** (temporarily empty examples) vs **two-shot** — document confusion cases.

For a **novel distractor** (e.g. new coloured puck), define a new prompt + 2–3 example crops + JSON answers, call `ask(..., examples=your_pairs)`, and require `confidence >= 0.8` else `skip_reason: low_confidence_novel`.

In [ ]:
ICL_DIR.mkdir(parents=True, exist_ok=True)
readme = ICL_DIR / "README.txt"
if not readme.exists():
    readme.write_text(
        "Place wide_end_up.png and narrow_end_up.png here for cap ICL.\n"
        "Create crops from your own run captures — do not commit private data if policy forbids.\n",
        encoding="utf-8",
    )
print("ICL folder:", ICL_DIR)
print("Existing PNGs:", list(ICL_DIR.glob("*.png")))

## 6. Full pipeline

In [ ]:
if RUN.exists() and any(RUN.iterdir()):
    raise SystemExit(
        f"Refusing to run: {RUN} must be empty. Pick a new TUTORIAL_RUN_ID or remove contents."
    )
RUN.mkdir(parents=True, exist_ok=True)

In [ ]:
# Live end-to-end — requires Isaac + Cosmos (several minutes)
pipeline = SIM / "run_sort_pipeline.sh"
host_py = os.environ.get("HOST_PYTHON", str(SIM / ".venv/bin/python"))
env = os.environ.copy()
env["HOST_PYTHON"] = host_py
subprocess.run(["bash", str(pipeline), str(RUN)], cwd=SIM, env=env, check=True)

In [ ]:
report_path = RUN / "execution_report.json"
if not report_path.is_file():
    raise FileNotFoundError("No execution report — inspect Isaac logs")

report = json.loads(report_path.read_text())
assert report.get("task_complete") is True, report.get("abort_reason")
assert report.get("verified_pickups") == 6
assert report.get("verified_placements") == 6
print("Pipeline success:", report_path)
print("Videos:", RUN / "sort_table.mp4", RUN / "sort_side.mp4")

## 7. When something fails

| Symptom | Inspect |
|---------|---------|
| No picks in plan | `sort_plan.json` skip reasons; `detections_raw.txt` |
| Pose rejected | `cube_pose_crops/*/measurement.json`; IoU/confidence gates |
| Incomplete stack | `execution_report.json`, Isaac stdout |
| Missing model | `COSMOS3_NANO_PATH`, Cosmos import errors |

Re-run physics only (after a good plan):

```bash
"$ISAAC/python.sh" sort_scene.py --run runs/<id> --physics-only
```

## Cleanup

- Stop/pause Brev instance.
- Keep large runs out of git (`runs/` is gitignored).
- Remove tutorial code experiments you do not intend to merge.